# Trips Table Audit

## Objective

Perform a data quality assessment of the Trips dataset before loading it into the Silver Layer.

The audit focuses on:

- Missing values
- Duplicate records
- Primary key validation
- Foreign key validation
- Data type validation
- Geospatial validation
- Business rule validation

In [1]:
import pandas as pd

## 1. Load Dataset

Load the Trips dataset into a Pandas DataFrame and perform an initial inspection.

In [2]:
trips = pd.read_csv(r"C:\Dataset\trips.csv")

## 2. Dataset Structure Review

Review:

- Number of records
- Number of columns
- Data types
- Overall dataset structure

In [3]:
trips.info

<bound method DataFrame.info of                trip_id  start_lat  start_lon    end_lat    end_lon  \
0        TRIP000000000  28.490285  76.935970  28.876145  77.116253   
1        TRIP000000001  28.612242  77.171701  28.756341  76.955711   
2        TRIP000000002  28.473643  77.293100  28.653529  77.138391   
3        TRIP000000003  28.490291  77.320918  28.553302  77.097586   
4        TRIP000000004  28.515472  77.187308  28.656728  76.905885   
...                ...        ...        ...        ...        ...   
2039995  TRIP001918149  28.457246  77.312875  28.405639  77.346642   
2039996  TRIP001901687  28.403420  77.060693  28.761567  76.960541   
2039997  TRIP001963717  28.721523  77.124873  28.611203  77.156742   
2039998  TRIP001933933  28.434026  77.005980  28.438185  77.130671   
2039999  TRIP001992570  28.415697  76.848753  28.733134  77.345366   

         distance_km            timestamp  road_id traffic_id weather_id  \
0              9.696     05-26-2023 18:21  RD00905 

In [4]:
trips.dtypes

trip_id             object
start_lat          float64
start_lon          float64
end_lat            float64
end_lon            float64
distance_km        float64
timestamp           object
road_id             object
traffic_id          object
weather_id          object
travel_time_min    float64
dtype: object

## 3. Missing Value Analysis

Identify missing values across all columns and evaluate their impact on data completeness.

In [5]:
trips.isnull().sum()

trip_id                0
start_lat              0
start_lon              0
end_lat                0
end_lon                0
distance_km            0
timestamp          30782
road_id            81119
traffic_id         82153
weather_id         61156
travel_time_min    20337
dtype: int64

## 4. Duplicate Record Analysis

Check for fully duplicated records that may have resulted from ingestion or processing issues.

In [6]:
trips.duplicated().sum()

np.int64(40000)

## 5. Primary Key Validation

Evaluate whether `trip_id` can serve as a reliable primary key by validating:

- NULL values
- Uniqueness
- Duplicate occurrences

In [7]:
trips['trip_id'].nunique()

2000000

In [8]:
trips['trip_id'].duplicated().sum()

np.int64(40000)

## 6. Foreign Key Validation

Review candidate foreign key columns and assess:

- Missing values
- Distinct values
- Referential integrity readiness

Candidate Foreign Keys:

- road_id
- traffic_id
- weather_id

In [9]:
trips['road_id'].nunique()

9641

In [10]:
trips['road_id'].duplicated().sum()

np.int64(2030358)

In [11]:
trips['traffic_id'].nunique()

145932

In [12]:
trips['traffic_id'].duplicated().sum()

np.int64(1894067)

In [13]:
trips['weather_id'].nunique()

145555

In [14]:
trips['weather_id'].duplicated().sum()

np.int64(1894444)

## 7. Geospatial Validation

Validate geographic coordinates against accepted ranges.

Expected Ranges:

Latitude:
-90° to +90°

Longitude:
-180° to +180°

The objective is to identify:

- Invalid coordinates
- Potential latitude/longitude swaps
- Data corruption issues

## 8. Start Location Validation

Review origin coordinates and identify records that violate latitude and longitude constraints.

In [15]:
trips[~trips['start_lat'].between(-90, 90)].count()

trip_id            17012
start_lat          17012
start_lon          17012
end_lat            17012
end_lon            17012
distance_km        17012
timestamp          16735
road_id            16332
traffic_id         16274
weather_id         16496
travel_time_min    16848
dtype: int64

In [16]:
trips[~trips['start_lat'].between(-180, 180)].count()

trip_id            3159
start_lat          3159
start_lon          3159
end_lat            3159
end_lon            3159
distance_km        3159
timestamp          3100
road_id            3028
traffic_id         3014
weather_id         3058
travel_time_min    3124
dtype: int64

In [17]:
trips[~trips['start_lon'].between(-90, 90)].count()

trip_id            21667
start_lat          21667
start_lon          21667
end_lat            21667
end_lon            21667
distance_km        21667
timestamp          21314
road_id            20784
traffic_id         20773
weather_id         21018
travel_time_min    21443
dtype: int64

In [18]:
trips[~trips['start_lon'].between(-180, 180)].count()

trip_id            12376
start_lat          12376
start_lon          12376
end_lat            12376
end_lon            12376
distance_km        12376
timestamp          12181
road_id            11899
traffic_id         11855
weather_id         12007
travel_time_min    12257
dtype: int64

## 9. Coordinate Integrity Assessment

Investigate potential coordinate corruption.

The assessment focuses on identifying records where latitude and longitude values may have been accidentally swapped during data generation or ingestion.

In [19]:
trips[(~trips['start_lat'].between(-90, 90) &
       trips['start_lat'].between(-180, 180) &
       trips['start_lon'].between(-90, 90))].count()

trip_id            4086
start_lat          4086
start_lon          4086
end_lat            4086
end_lon            4086
distance_km        4086
timestamp          4012
road_id            3922
traffic_id         3911
weather_id         3957
travel_time_min    4054
dtype: int64

## 10. Destination Coordinate Validation

Validate destination coordinates and assess overall coordinate quality.

In [20]:
trips[~trips['end_lat'].between(-90, 90)].count()

trip_id            0
start_lat          0
start_lon          0
end_lat            0
end_lon            0
distance_km        0
timestamp          0
road_id            0
traffic_id         0
weather_id         0
travel_time_min    0
dtype: int64

In [21]:
trips[~trips['end_lat'].between(-180, 180)].count()

trip_id            0
start_lat          0
start_lon          0
end_lat            0
end_lon            0
distance_km        0
timestamp          0
road_id            0
traffic_id         0
weather_id         0
travel_time_min    0
dtype: int64

In [22]:
trips[~trips['end_lon'].between(-90, 90)].count()

trip_id            0
start_lat          0
start_lon          0
end_lat            0
end_lon            0
distance_km        0
timestamp          0
road_id            0
traffic_id         0
weather_id         0
travel_time_min    0
dtype: int64

In [23]:
trips[~trips['end_lon'].between(-180, 180)].count()

trip_id            0
start_lat          0
start_lon          0
end_lat            0
end_lon            0
distance_km        0
timestamp          0
road_id            0
traffic_id         0
weather_id         0
travel_time_min    0
dtype: int64

## 11. Timestamp Validation

Review timestamp values for:

- Data type correctness
- Format consistency
- Temporal analysis readiness

The objective is to prepare the column for future date-based validation and analytics.

In [24]:
trips['timestamp']

0             05-26-2023 18:21
1                   99-99-9999
2            01-Jul-2023 11:54
3          2023-13-45 25:99:00
4               21/01/23 10:20
                  ...         
2039995    2023-11-16 07:49:02
2039996      16-Jul-2023 02:45
2039997    2023-10-30 23:18:07
2039998       02-20-2023 14:56
2039999       12-21-2023 06:39
Name: timestamp, Length: 2040000, dtype: object

## 12. Travel Time Validation

Validate travel duration measurements.

The objective is to identify:

- Missing values
- Invalid values
- Business rule violations

In [25]:
trips[trips['travel_time_min'] <0 ]

,trip_id,start_lat,start_lon,end_lat,end_lon,distance_km,timestamp,road_id,traffic_id,weather_id,travel_time_min


In [26]:
trips[trips['travel_time_min'] == 0 ]

,trip_id,start_lat,start_lon,end_lat,end_lon,distance_km,timestamp,road_id,traffic_id,weather_id,travel_time_min


## 13. Audit Findings Summary

### Key Findings

- 30,782 missing timestamp values
- 81,119 missing road_id values
- 82,153 missing traffic_id values
- 61,156 missing weather_id values
- 20,337 missing travel_time_min values
- 40,000 fully duplicated records
- Duplicate trip_id values detected
- Invalid start coordinate values detected
- 4086 Potential latitude/longitude swaps identified
- Timestamp stored as object datatype

### Candidate Primary Key Status

❌ trip_id cannot currently be used as a primary key.

### Candidate Foreign Keys

- road_id
- traffic_id
- weather_id

### Recommended Silver Layer Actions

- Remove duplicate records
- Resolve duplicate trip_id values
- Handle missing values
- Validate foreign key relationships
- Convert timestamp to datetime format
- Investigate coordinate inconsistencies
- Validate geospatial integrity